### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [1]:
import os 
from langchain_groq import ChatGroq

os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")

model = ChatGroq(
    model="qwen/qwen3.6-27b"
)

response = model.invoke("Why Do Parraot Talk?")
response

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - **Question:** "Why Do Parraot Talk?" (Note: "Parraot" is a typo for "Parrot")\n   - **Core Subject:** Parrots and their ability to talk/mimic human speech\n   - **Key Question:** Why do they do it? (Evolutionary, biological, social, cognitive reasons)\n\n2.  **Identify Key Concepts/Scientific Facts:**\n   - Parrots don\'t actually "talk" in the human sense (they don\'t understand language like humans do)\n   - They mimic sounds, including human speech\n   - Biological basis: Specialized vocal anatomy (syrinx, tongue, beak, brain structures like the pallium)\n   - Evolutionary/Social reasons: Social bonding, flock communication, attention-seeking, survival adaptation\n   - Cognitive aspects: High intelligence, pattern recognition, associative learning\n   - Context: Wild vs. captive parrots\n\n3.  **Structure the Response:**\n   - Clarify the misconception (they mimic, don\'t truly "talk")\n  

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools = model.bind_tools([get_weather])



In [ ]:
response = model_with_tools.invoke("Whats the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  Identify the user\'s intent: The user wants to know the current weather in Boston.\n2.  Identify available tools: `get_weather(location: string)`\n3.  Extract parameters: location = "Boston"\n4.  Call the function: `get_weather(location="Boston")`\n5.  Formulate response based on the tool\'s output. (I will simulate the tool call first)\nWait, I need to actually call the tool. I\'ll generate the tool call.✅\nLet\'s do it. \nParameters: `{"location": "Boston"}`.\nFunction: `get_weather`.\nOutput will be handled after the tool returns. I will generate the tool call now. \nSelf-Correction/Verification: The prompt exactly matches the tool definition. I will proceed. \n`print(get_weather(location="Boston"))` -> I\'ll format it as a tool call. \nDone. \nProceeds. \n[Output Generation] -> Tool Call. \n`{"name": "get_weather", "arguments": {"location": "Boston"}}`\nWait, the system expects me to output the tool call in 

### Tool Execution Loops

In [7]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

It's sunny in Boston.
